# PART 04 · 4장 상관분석과 회귀분석 — 연습 문제 정답

교재 376쪽 연습 문제의 정답 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

## 공통 준비

In [1]:
import pandas as pd
import statsmodels.api as sm

path = (
    'data/part04_type3/correlation_regression_ad_orders/'
    'ad_orders.csv'
)
df = pd.read_csv(path)
print(df.shape)
print(df.head())

(80, 4)
         date  ad_budget  orders  is_weekend
0  2026-01-01      139.5     326           0
1  2026-01-02      114.6     323           0
2  2026-01-03       80.5     256           1
3  2026-01-04       41.8     160           1
4  2026-01-05       40.6     145           0


## 문제 01

다중회귀에서 is_weekend의 회귀계수 18.9344를 가장 정확하게 해석하시오.

**정답 코드**

In [2]:
X_multi = sm.add_constant(df[['ad_budget', 'is_weekend']])
model_multi = sm.OLS(df['orders'], X_multi).fit()
print(model_multi.params.round(4))

const         83.0955
ad_budget      2.0163
is_weekend    18.9344
dtype: float64


**간단 해설**

0·1 변수의 계수는 다른 독립변수가 같을 때 1인 집단과 0인 집단의 평균 예측값 차이다.

## 문제 02

ad_orders 데이터의 date에서 day_of_month를 만든다. ad_budget, is_weekend, day_of_month를 독립변수로 다중선형 회귀모형을 적합한 뒤 유의수준 0.05에서 유의한 독립변수만으로 모형을 다시 적합하시오. 다음 값을 구하시오.

- 광고비가 90이고 평일인 경우의 예측 주문 수

**정답 코드**

In [3]:
df['date'] = pd.to_datetime(df['date'])
df['day_of_month'] = df['date'].dt.day

features = [
    'ad_budget',
    'is_weekend',
    'day_of_month'
]
X_full = sm.add_constant(
    df[features],
    has_constant='add'
)
y = df['orders']
full_model = sm.OLS(y, X_full).fit()
print(full_model.pvalues.round(4))

alpha = 0.05
feature_pvalues = full_model.pvalues.drop('const')
sig_vars = feature_pvalues[
    feature_pvalues < alpha
].index.tolist()

X_refit = sm.add_constant(
    df[sig_vars],
    has_constant='add'
)
refit_model = sm.OLS(y, X_refit).fit()
print('selected_variables:', sig_vars)
print(refit_model.params.round(4))

new_data = pd.DataFrame({
    'const': [1],
    'ad_budget': [90],
    'is_weekend': [0]
})
predicted = refit_model.predict(new_data)
print('predicted_orders:', round(float(predicted.iloc[0]), 4))

const           0.0000
ad_budget       0.0000
is_weekend      0.0001
day_of_month    0.4511
dtype: float64
selected_variables: ['ad_budget', 'is_weekend']
const         83.0955
ad_budget      2.0163
is_weekend    18.9344
dtype: float64
predicted_orders: 264.5581


**간단 해설**

day_of_month는 유의수준 0.05에서 선택되지 않는다. 재적합 모형에는 ad_budget과 is_weekend만 사용하며, 광고비가 90이고 평일일 때 예측 주문 수는 264.5581이다.

---

**정답** 1. 광고비가 같을 때 주말은 평일보다 주문 수가 평균적으로 18.9344 높다, 2. 264.5581